In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

2026-04-28 05:16:32.085405: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777353392.098815      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777353392.103031      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 05:16:32.119309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
ROOT_DIR = '/kaggle/input/'
AUTOTUNE = tf.data.experimental.AUTOTUNE
HEIGHT, WIDTH = 240, 240 # EfficientNet B0 input shape
BATCH_SIZE = 32

In [3]:
def _parse_function(example, feature_description):
    parsed_example = tf.io.parse_single_example(example, feature_description)
    image = tf.io.decode_jpeg(parsed_example['image'], channels=3)
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, (HEIGHT, WIDTH))
    if 'target' in feature_description:
        target = parsed_example['target']
        return image, target
    return image, parsed_example['image_name']


def load_data(path, batch_size=32, train=True):
    filenames = tf.io.gfile.glob(path)
    dataset = tf.data.TFRecordDataset(filenames, num_parallel_reads=tf.data.experimental.AUTOTUNE)
    if train:
        feature_description = {
            'image': tf.io.FixedLenFeature([], tf.string),
            'image_name': tf.io.FixedLenFeature([], tf.string),
            'target': tf.io.FixedLenFeature([], tf.int64)
        }
    else:
        feature_description = {
            'image': tf.io.FixedLenFeature([], tf.string),
            'image_name': tf.io.FixedLenFeature([], tf.string)
        }
    parsed_dataset = dataset.map(lambda x: _parse_function(x, feature_description))
    return parsed_dataset.batch(batch_size).prefetch(AUTOTUNE)

In [4]:
# from sklearn.model_selection import train_test_split
# train_filenames = tf.io.gfile.glob(ROOT_DIR + 'cassava-leaf-disease-classification/train_tfrecords/' + 'ld_train*.tfrec')
# train_filenames, val_filenames = train_test_split(train_filenames, test_size=0.3, random_state=37)
# val_set = load_data(val_filenames, batch_size=BATCH_SIZE, train=False)

In [5]:
test_set = load_data(ROOT_DIR + 'cassava-leaf-disease-classification/test_tfrecords/ld_test*.tfrec', batch_size=BATCH_SIZE, train=False)

2026-04-28 05:16:38.739380: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
model = tf.keras.models.load_model(ROOT_DIR + 'cassava-training/effnetb1_finetuned.h5')

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/kaggle/input/cassava-training/effnetb1_finetuned.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [7]:
test_IDs, predictions = [], []
for item in test_set:
    pred = np.argmax(model.predict_on_batch(item[0]), axis=1)
    ID_batch = [fname.decode() for fname in item[1].numpy()]
    test_IDs.extend(ID_batch)
    predictions.extend(pred)

2026-04-28 05:16:39.155302: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144


NameError: name 'model' is not defined

In [8]:
submission_df = pd.DataFrame({'image_id': test_IDs, 'label': predictions})
submission_df.to_csv('submission.csv', index=False)